# Dunnhumby 구매이웃 기반 N/V 조건부 M2 — 10시드 최종 test

현재 채택한 v2 구조를 바꾸지 않고 seed 42~51에서 M1@64와 M2를 동일 조건으로 새로 학습합니다.

- 학습자료: 기존 train + validation 병합
- 평가: 각 seed·모형의 100 epoch 최종 checkpoint에서 test 한 번
- validation·조기종료·epoch 선택 없음, holdout 미사용
- binary graph, uniform negative sampling, 표본 가중 없는 plain BPR
- 시드별 결과와 10시드 평균·표준편차·95% t 구간 저장
- 중단 후 재실행하면 완료 arm은 test를 다시 평가하지 않고 저장 결과를 재사용


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'fd291d59e79b47d29f7e090b5a8d039716f6c32b'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('코드 고정 완료:', REVIEWED_SHA)


In [ ]:
import json
from pathlib import Path
import torch
from lightgcn_clv_neighbor_conditioned_id_transform_test10 import (
    configure_neighbor_conditioned_test10_run,
    preflight_summary,
    run_neighbor_conditioned_test10,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_neighbor_conditioned_test10_run(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_neighbor_conditioned_id_transform_test10_v1'
    ),
)
summary = preflight_summary(cfg)
assert summary['seeds'] == list(range(42, 52))
assert summary['trained_models'] == [
    'm1_64', 'm2_neighbor_conditioned_id_transform'
]
assert summary['validation_selection'] is False
assert summary['holdout_evaluation'] is False
assert summary['fixed_boundaries']['graph'] == 'binary'
assert summary['fixed_boundaries']['negative_sampling'] == 'uniform'
assert summary['m2']['embedding_dim'] == 64
assert summary['m2']['transform_rank'] == 4
assert summary['m2']['rho'] == 0.05
print(json.dumps(summary, ensure_ascii=False, indent=2))


## 저장된 진행상태 확인

학습을 시작하지 않습니다. 런타임 재연결 후 완료 arm과 저장 epoch를 확인할 때 사용합니다.


In [ ]:
progress_files = sorted(Path(cfg.out_dir).glob('progress/**/progress.json'))
if not progress_files:
    print('아직 저장된 progress.json이 없습니다.')
else:
    for progress_path in progress_files:
        print(progress_path)
        print(json.dumps(json.loads(progress_path.read_text()), ensure_ascii=False, indent=2))


## 10시드 일괄 실행

아래 셀 하나가 10개 seed × 2개 모형을 순차 실행합니다. 런타임이 끊기면 위에서부터 다시 실행한 뒤 이 셀을 다시 실행하세요.


In [ ]:
result_df = run_neighbor_conditioned_test10(cfg)


In [ ]:
from IPython.display import display

absolute_summary = result_df.attrs['absolute_summary'].copy()
paired_seed = result_df.attrs['paired_seed'].copy()
paired_summary = result_df.attrs['paired_summary'].copy()
reading = dict(result_df.attrs['descriptive_reading'])
paths = dict(result_df.attrs['result_paths'])
display_df = result_df.copy()
display_df.attrs = {}

print('seed별 절대지표 (20행):')
display(display_df)
print('10시드 절대지표 평균·표준편차·95% 구간:')
display(absolute_summary)
print('동일 seed M1@64 대비 개별 차이:')
display(paired_seed)
print('동일 seed M1@64 대비 10시드 평균 차이:')
display(paired_summary)
print('판독:')
print(json.dumps(reading, ensure_ascii=False, indent=2))
print('결과 파일:')
print(json.dumps(paths, ensure_ascii=False, indent=2))
